# Feast Operator with RBAC Configuration
## Objective

This demo provides a reference implementation of a runbook on how to enable Role-Based Access Control (RBAC) for Feast using the Feast Operator with the Kubernetes authentication type.

The demo steps include deploying the Feast Operator, creating Feast instances with server components (registry, offline store, online store), and client examples within a Kubernetes environment. The goal is to ensure secure access control for Feast instances deployed by the Feast Operator.
 
Please read these reference documents for understanding the Feast RBAC framework.
- [RBAC Architecture](https://docs.feast.dev/v/master/getting-started/architecture/rbac) 
- [RBAC Permission](https://docs.feast.dev/v/master/getting-started/concepts/permission).
- [RBAC Authorization Manager](https://docs.feast.dev/v/master/getting-started/components/authz_manager)


## Prerequisites
* Kubernetes Cluster
* [kubectl](https://kubernetes.io/docs/tasks/tools/#kubectl) Kubernetes CLI tool.

## Install Prerequisites

The following commands install and configure all the prerequisites on a MacOS environment. You can find the
equivalent instructions on the offical documentation pages:
* Install the `kubectl` cli.
* Install Kubernetes and Container runtime (e.g. [Colima](https://github.com/abiosoft/colima)).
  * Alternatively, authenticate to an existing Kubernetes or OpenShift cluster.
  
```bash
brew install colima kubectl
colima start -r containerd -k -m 3 -d 100 -c 2 --cpu-type max -a x86_64
colima list
```

In [120]:
!kubectl create ns feast
!kubectl config set-context --current --namespace feast

namespace/feast created
Context "kind-kind" modified.


Validate the cluster setup:

In [30]:
!kubectl get ns feast

NAME    STATUS   AGE
feast   Active   8s


## Install the Feast Operator

In [121]:
## Use this install command from a release branch (e.g. 'v0.43-branch')
!kubectl apply -f ../../infra/feast-operator/dist/install.yaml

## OR, for the latest code/builds, use one the following commands from the 'master' branch
# !make -C ../../infra/feast-operator install deploy IMG=quay.io/feastdev-ci/feast-operator:develop FS_IMG=quay.io/feastdev-ci/feature-server:develop
# !make -C ../../infra/feast-operator install deploy IMG=quay.io/feastdev-ci/feast-operator:$(git rev-parse HEAD) FS_IMG=quay.io/feastdev-ci/feature-server:$(git rev-parse HEAD)

!kubectl wait --for=condition=available --timeout=5m deployment/feast-operator-controller-manager -n feast-operator-system

namespace/feast-operator-system created
customresourcedefinition.apiextensions.k8s.io/featurestores.feast.dev created
serviceaccount/feast-operator-controller-manager created
role.rbac.authorization.k8s.io/feast-operator-leader-election-role created
clusterrole.rbac.authorization.k8s.io/feast-operator-featurestore-editor-role created
clusterrole.rbac.authorization.k8s.io/feast-operator-featurestore-viewer-role created
clusterrole.rbac.authorization.k8s.io/feast-operator-manager-role created
clusterrole.rbac.authorization.k8s.io/feast-operator-metrics-auth-role created
clusterrole.rbac.authorization.k8s.io/feast-operator-metrics-reader created
rolebinding.rbac.authorization.k8s.io/feast-operator-leader-election-rolebinding created
clusterrolebinding.rbac.authorization.k8s.io/feast-operator-manager-rolebinding created
clusterrolebinding.rbac.authorization.k8s.io/feast-operator-metrics-auth-rolebinding created
service/feast-operator-controller-manager-metrics-service created
deployment.ap

## Deployment Architecture
The primary objective of this runbook is to guide the deployment of Feast services with RBAC

In this notebook, we will deploy a distributed topology of Feast services, which includes:

* `Registry Server`: Handles metadata storage for feature definitions.
* `Online Store Server`: Uses the `Registry Server` to query metadata and is responsible for low-latency serving of features.
* `Offline Store Server`: Uses the `Registry Server` to query metadata and provides access to batch data for historical feature retrieval.
* `Kubernetes` Authentication types for RBAC Configuration for Feast resources.
* Setting update client RBAC examples to test Feast RBAC based on roles assigned.


## Install the Feast services via FeatureStore CR
Next, we'll use the running Feast Operator to install the feast services with Server components online, offline, registry with kubernetes Authorization set. Apply the included [reference deployment](../../infra/feast-operator/config/samples/v1alpha1_featurestore_kubernetes_auth.yaml) to install and configure Feast with kubernetes Authorization .

In [122]:
!cat ../../infra/feast-operator/config/samples/v1alpha1_featurestore_kubernetes_auth.yaml
!kubectl apply -f ../../infra/feast-operator/config/samples/v1alpha1_featurestore_kubernetes_auth.yaml -n feast

apiVersion: feast.dev/v1alpha1
kind: FeatureStore
metadata:
  name: sample-kubernetes-auth
spec:
  feastProject: feast_rbac
  authz:
    kubernetes:
      roles:
        - feast-writer
        - feast-reader
  services:
    offlineStore:
      server: {}
    onlineStore:
      server: {}
    registry:
      local:
        server: {}
    ui: {}
featurestore.feast.dev/sample-kubernetes-auth created


## Validate the running FeatureStore deployment
Validate the deployment status.

In [123]:
!kubectl get all
!kubectl wait --for=condition=available --timeout=8m deployment/feast-sample-kubernetes-auth

NAME                                                READY   STATUS    RESTARTS   AGE
pod/feast-sample-kubernetes-auth-774f6df8df-q7v7f   0/4     Running   0          13s

NAME                                            TYPE        CLUSTER-IP      EXTERNAL-IP   PORT(S)   AGE
service/feast-sample-kubernetes-auth-offline    ClusterIP   10.96.75.204    <none>        80/TCP    13s
service/feast-sample-kubernetes-auth-online     ClusterIP   10.96.234.54    <none>        80/TCP    13s
service/feast-sample-kubernetes-auth-registry   ClusterIP   10.96.130.144   <none>        80/TCP    13s
service/feast-sample-kubernetes-auth-ui         ClusterIP   10.96.165.234   <none>        80/TCP    13s

NAME                                           READY   UP-TO-DATE   AVAILABLE   AGE
deployment.apps/feast-sample-kubernetes-auth   0/1     1            0           13s

NAME                                                      DESIRED   CURRENT   READY   AGE
replicaset.apps/feast-sample-kubernetes-auth-774f

Validate that the FeatureStore CR is in a `Ready` state.

In [35]:
!kubectl get feast

NAME                     STATUS   AGE
sample-kubernetes-auth   Ready    81s


## Configure the RBAC Permissions
we have defined permission in `permissions_apply.py`.

In [36]:
#view the permissions  
!cat  permissions_apply.py

from feast.feast_object import ALL_RESOURCE_TYPES
from feast.permissions.action import READ, AuthzedAction, ALL_ACTIONS
from feast.permissions.permission import Permission
from feast.permissions.policy import RoleBasedPolicy

admin_roles = ["feast-writer"]
user_roles = ["feast-reader"]

user_perm = Permission(
    name="feast_user_permission",
    types=ALL_RESOURCE_TYPES,
    policy=RoleBasedPolicy(roles=user_roles),
    actions=[AuthzedAction.DESCRIBE] + READ
)

admin_perm = Permission(
    name="feast_admin_permission",
    types=ALL_RESOURCE_TYPES,
    policy=RoleBasedPolicy(roles=admin_roles),
    actions=ALL_ACTIONS
)


In [124]:
# Copy the Permissions to the pods under feature_repo directory
!kubectl cp permissions_apply.py $(kubectl get pods -l 'feast.dev/name=sample-kubernetes-auth' -ojsonpath="{.items[*].metadata.name}"):/feast-data/feast_rbac/feature_repo -c online

In [125]:
#view the feature_store.yaml configuration 
!kubectl exec deploy/feast-sample-kubernetes-auth -itc online -- cat feature_store.yaml

E0227 14:49:52.453894   50390 websocket.go:296] Unknown stream id 1, discarding message
project: feast_rbac
provider: local
offline_store:
    type: dask
online_store:
    path: /feast-data/online_store.db
    type: sqlite
registry:
    path: /feast-data/registry.db
    registry_type: file
auth:
    type: kubernetes
entity_key_serialization_version: 3


## Apply the Permissions and Feast Object to Registry

In [126]:
!kubectl exec deploy/feast-sample-kubernetes-auth -itc online -- feast apply

<jemalloc>: MADV_DONTNEED does not work (memset will be used instead)
<jemalloc>: (This is the expected behaviour if you are running under QEMU)
/opt/app-root/lib64/python3.11/site-packages/feast/feature_view.py:48: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity '__dummy'.
  DUMMY_ENTITY = Entity(
/opt/app-root/lib64/python3.11/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "vector_enabled" in "SqliteOnlineStoreConfig" shadows an attribute in parent "VectorStoreConfig"
  warnings.warn(
/opt/app-root/lib64/python3.11/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "vector_len" in "SqliteOnlineStoreConfig" shadows an attribute in parent "VectorStoreConfig"
  warnings.warn(
/feast-data/feast_rbac/feature_repo/example_repo.py:27: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity 'driver'.
  driver = Entity(

**List the applied permission details permissions on Feast Resources.**

In [127]:
!kubectl exec deploy/feast-sample-kubernetes-auth -itc online -- feast permissions list-roles
!kubectl exec deploy/feast-sample-kubernetes-auth -itc online -- feast permissions list
!kubectl exec deploy/feast-sample-kubernetes-auth -itc online -- feast permissions describe feast_admin_permission
!kubectl exec deploy/feast-sample-kubernetes-auth -itc online -- feast permissions describe feast_user_permission

<jemalloc>: MADV_DONTNEED does not work (memset will be used instead)
<jemalloc>: (This is the expected behaviour if you are running under QEMU)
/opt/app-root/lib64/python3.11/site-packages/feast/feature_view.py:48: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity '__dummy'.
  DUMMY_ENTITY = Entity(
/opt/app-root/lib64/python3.11/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "vector_enabled" in "SqliteOnlineStoreConfig" shadows an attribute in parent "VectorStoreConfig"
  warnings.warn(
/opt/app-root/lib64/python3.11/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "vector_len" in "SqliteOnlineStoreConfig" shadows an attribute in parent "VectorStoreConfig"
  warnings.warn(
+--------------+
| ROLE NAME    |
+==============+
| feast-reader |
+--------------+
| feast-writer |
+--------------+
<jemalloc>: MADV_DONTNEED does not work (memset will be used instead)
<jemalloc>:

## Feast Client with RBAC
### RBAC Kubernetes Authentication
This Feast **Role-Based Access Control (RBAC)** in Kubernetes support authentication  **inside a Kubernetes pod** and **outside a pod** when running a local script.
### Inside a Kubernetes Pod
Feast automatically retrieves the Kubernetes ServiceAccount token from:
```
/var/run/secrets/kubernetes.io/serviceaccount/token
```
This means:
- No manual configuration is needed inside a pod.
- The token is mounted automatically and used for authentication.
- Developer just need create the binding with role and service account accordingly.
- Code Reference:  
[Feast Kubernetes Auth Client Manager (Pod Token Usage)](https://github.com/feast-dev/feast/blob/master/sdk/python/feast/permissions/client/kubernetes_auth_client_manager.py#L15) 
- See the example will use service account from pod [Example](https://github.com/feast-dev/feast/blob/master/examples/rbac-remote/client/k8s/)

### Outside a Kubernetes Pod (Local Machine)
If running Feast outside of Kubernetes, authentication requires setting the token manually:
```sh
export LOCAL_K8S_TOKEN="your-service-account-token"
```
Feast will use this token for authentication.

Reference:  
[Feast Authentication via `LOCAL_K8S_TOKEN`](https://github.com/feast-dev/feast/blob/master/sdk/python/feast/permissions/client/kubernetes_auth_client_manager.py#L50)

###  Feature Store settings

In [43]:
!cat client/feature_store.yaml

project: feast_rbac
provider: local
offline_store:
    host: feast-sample-kubernetes-auth-offline.feast.svc.cluster.local
    type: remote
    port: 80
online_store:
    path: http://feast-sample-kubernetes-auth-online.feast.svc.cluster.local:80
    type: remote
registry:
    path: feast-sample-kubernetes-auth-registry.feast.svc.cluster.local:80
    registry_type: remote
auth:
    type: kubernetes
entity_key_serialization_version: 3


**The Operator creates the client ConfigMap containing the feature_store.yaml. We can retrieve it and port froward to local**

In [128]:
!kubectl get configmap feast-sample-kubernetes-auth-client -n feast -o jsonpath='{.data.feature_store\.yaml}' 

project: feast_rbac
provider: local
offline_store:
    host: feast-sample-kubernetes-auth-offline.feast.svc.cluster.local
    type: remote
    port: 80
online_store:
    path: http://feast-sample-kubernetes-auth-online.feast.svc.cluster.local:80
    type: remote
registry:
    path: feast-sample-kubernetes-auth-registry.feast.svc.cluster.local:80
    registry_type: remote
auth:
    type: kubernetes
entity_key_serialization_version: 3


### The function below is executed to support the preparation of client testing.

Run Port Forwarding for All Services for local testing 

In [129]:
import subprocess

# Define services and their local ports
services = {
    "offline_store": ("feast-sample-kubernetes-auth-offline", 8081),
    "online_store": ("feast-sample-kubernetes-auth-online", 8082),
    "registry": ("feast-sample-kubernetes-auth-registry", 8083),
}

# Start port-forwarding for each service
port_forward_processes = {}
for name, (service, local_port) in services.items():
    cmd = f"kubectl port-forward svc/{service} -n feast {local_port}:80"
    process = subprocess.Popen(cmd, shell=True)
    port_forward_processes[name] = process
    print(f"Port forwarding {service} -> localhost:{local_port}")

Port forwarding feast-sample-kubernetes-auth-offline -> localhost:8081
Port forwarding feast-sample-kubernetes-auth-online -> localhost:8082
Port forwarding feast-sample-kubernetes-auth-registry -> localhost:8083


Function to retrieve a Kubernetes service account token and set it as an environment variable

In [130]:
import subprocess
import os

def get_k8s_token(service_account):
    namespace = "feast"

    if not service_account:
        raise ValueError("Service account name is required.")

    result = subprocess.run(
        ["kubectl", "create", "token", service_account, "-n", namespace],
        capture_output=True, text=True, check=True
    )

    token = result.stdout.strip()

    if not token:
        return None  # Silently return None if token retrieval fails

    os.environ["LOCAL_K8S_TOKEN"] = token
    return "Token Retrieved: ***** (hidden for security)"


**Generating training data. The following test functions were copied from the `test_workflow.py` template but we added `try` blocks to print only 
the relevant error messages, since we expect to receive errors from the permission enforcement modules.**

In [135]:
from feast import FeatureStore

def fetch_historical_features_entity_df(store: FeatureStore, for_batch_scoring: bool):
    try:
        entity_df = pd.DataFrame.from_dict(
            {
                "driver_id": [1001, 1002, 1003],
                "event_timestamp": [
                    datetime(2021, 4, 12, 10, 59, 42),
                    datetime(2021, 4, 12, 8, 12, 10),
                    datetime(2021, 4, 12, 16, 40, 26),
                ],
                "label_driver_reported_satisfaction": [1, 5, 3],
                # values we're using for an on-demand transformation
                "val_to_add": [1, 2, 3],
                "val_to_add_2": [10, 20, 30],
            }
        )
        if for_batch_scoring:
            entity_df["event_timestamp"] = pd.to_datetime("now", utc=True)

        training_df = store.get_historical_features(
            entity_df=entity_df,
            features=[
                "driver_hourly_stats:conv_rate",
                "driver_hourly_stats:acc_rate",
                "driver_hourly_stats:avg_daily_trips",
                "transformed_conv_rate:conv_rate_plus_val1",
                "transformed_conv_rate:conv_rate_plus_val2",
            ],

        ).to_df()
        print(training_df.head())

    except Exception as e:
        print(f"An error occurred while fetching historical features: {e}")


def fetch_online_features(store, source: str = ""):
    try:
        entity_rows = [
            # {join_key: entity_value}
            {
                "driver_id": 1001,
                "val_to_add": 1000,
                "val_to_add_2": 2000,
            },
            {
                "driver_id": 1002,
                "val_to_add": 1001,
                "val_to_add_2": 2002,
            },
        ]
        if source == "feature_service":
            features_to_fetch = store.get_feature_service("driver_activity_v1")
        elif source == "push":
            features_to_fetch = store.get_feature_service("driver_activity_v3")
        else:
            features_to_fetch = [
                "driver_hourly_stats:acc_rate",
                "transformed_conv_rate:conv_rate_plus_val1",
                "transformed_conv_rate:conv_rate_plus_val2",
            ]
        returned_features = store.get_online_features(
            features=features_to_fetch,
            entity_rows=entity_rows,
        ).to_dict()
        for key, value in sorted(returned_features.items()):
            print(key, " : ", value)

    except Exception as e:
        print(f"An error occurred while fetching online features: {e}")

## Setting Up and Testing Feast Users
The steps below will:
- Create **three different ServiceAccounts** for Feast.
- Assign appropriate **RoleBindings** for access control.
- Retrieve and use the **ServiceAccount token** for authentication.
- Run the test cases to validate RBAC.

## Test Cases
| User Type       | ServiceAccount               | RoleBinding Assigned | Expected Behavior in output                                |
|----------------|-----------------------------|----------------------|------------------------------------------------------------|
| **Read-Only**  | `feast-user-sa`              | `feast-reader`       | Can **read** from the feature store, but **cannot write**. |
| **Unauthorized** | `feast-unauthorized-user-sa` | _None_               | **Access should be denied** in `test.py`.                  |
| **Admin**      | `feast-admin-sa`             | `feast-writer`       | Can **read and write** feature store data.                 |

### Setup & Test Read-Only Feast User (serviceaccount: feast-user-sa, role: feast-reader)

**Step 1: Create the ServiceAccount and Role Binding**

In [131]:
# Step 1: Create the ServiceAccount
!echo "Creating ServiceAccount: feast-user-sa"
!kubectl create serviceaccount feast-user-sa -n feast

# Step 2: Assign RoleBinding (Read-Only Access for Feast)
!echo "Assigning Read-Only RoleBinding: feast-user-rolebinding"
!kubectl create rolebinding feast-user-rolebinding --role=feast-reader --serviceaccount=feast:feast-user-sa -n feast

Creating ServiceAccount: feast-user-sa
serviceaccount/feast-user-sa created
Assigning Read-Only RoleBinding: feast-user-rolebinding
rolebinding.rbac.authorization.k8s.io/feast-user-rolebinding created


**Step 2: Set the Token**

In [132]:
get_k8s_token("feast-user-sa")

'Token Retrieved: ***** (hidden for security)'

**Step 3:Test function  features from offline, online and materialize_incremental etc**

In [137]:
!echo "Running Feast RBAC test for Read only User..."
from feast.data_source import PushMode
from feast import FeatureStore
from datetime import datetime
import pandas as pd

try:

    store = FeatureStore(repo_path="client")

    print("\n--- Historical features for training ---")
    fetch_historical_features_entity_df(store, for_batch_scoring=False)

    print("\n--- Historical features for batch scoring ---")
    fetch_historical_features_entity_df(store, for_batch_scoring=True)

    try:
        print("\n--- Load features into online store/materialize_incremental ---")
        feature_views= store.list_feature_views()
        if not feature_views:
            raise PermissionError("No access to feature-views or no feature-views available.")
        store.materialize_incremental(end_date=datetime.now())
    except PermissionError as pe:
        print(f"Permission error: {pe}")
    except Exception as e:
        print(f"An occurred while performing materialize incremental: {e}")

    print("\n--- Online features ---")
    fetch_online_features(store)

    print("\n--- Online features retrieved (instead) through a feature service---")
    fetch_online_features(store, source="feature_service")

    print(
        "\n--- Online features retrieved (using feature service v3, which uses a feature view with a push source---"
    )
    fetch_online_features(store, source="push")

    print("\n--- Simulate a stream event ingestion of the hourly stats df ---")
    event_df = pd.DataFrame.from_dict(
        {
            "driver_id": [1001],
            "event_timestamp": [datetime.now()],
            "created": [datetime.now()],
            "conv_rate": [1.0],
            "acc_rate": [1.0],
            "avg_daily_trips": [1000],
        }
    )
    store.push("driver_stats_push_source", event_df, to=PushMode.ONLINE_AND_OFFLINE)

    print("\n--- Online features again with updated values from a stream push---")
    fetch_online_features(store, source="push")

except Exception as e:
    print(f"An error occurred: {e}")


Running Feast RBAC test for Read only User...

--- Historical features for training ---
Handling connection for 8083
Handling connection for 8081
   driver_id           event_timestamp  label_driver_reported_satisfaction  \
0       1001 2021-04-12 10:59:42+00:00                                   1   
1       1002 2021-04-12 08:12:10+00:00                                   5   
2       1003 2021-04-12 16:40:26+00:00                                   3   

   val_to_add  val_to_add_2  conv_rate  acc_rate  avg_daily_trips  \
0           1            10   0.274426  0.314969              971   
1           2            20   0.742959  0.215825              723   
2           3            30   0.466147  0.412917              247   

   conv_rate_plus_val1  conv_rate_plus_val2  
0             1.274426            10.274426  
1             2.742959            20.742959  
2             3.466147            30.466147  

--- Historical features for batch scoring ---
Handling connection for 8081
   d

  0%|                                                                         | 0/5 [00:00<?, ?it/s]


An occurred while performing materialize incremental: 

--- Online features ---
Handling connection for 8082
acc_rate  :  [None, None]
conv_rate_plus_val1  :  [None, None]
conv_rate_plus_val2  :  [None, None]
driver_id  :  [1001, 1002]

--- Online features retrieved (instead) through a feature service---
Handling connection for 8082
conv_rate  :  [None, None]
conv_rate_plus_val1  :  [None, None]
conv_rate_plus_val2  :  [None, None]
driver_id  :  [1001, 1002]

--- Online features retrieved (using feature service v3, which uses a feature view with a push source---
Handling connection for 8082
acc_rate  :  [None, None]
avg_daily_trips  :  [None, None]
conv_rate  :  [None, None]
conv_rate_plus_val1  :  [None, None]
conv_rate_plus_val2  :  [None, None]
driver_id  :  [1001, 1002]

--- Simulate a stream event ingestion of the hourly stats df ---
An error occurred: 


### Setup & Test Unauthorized Feast User (serviceaccount: feast-unauthorized-user-sa, role: None)

In [147]:
# Create the ServiceAccount (Without RoleBinding)
!echo "Creating Unauthorized ServiceAccount: feast-unauthorized-user-sa"
!kubectl create serviceaccount feast-unauthorized-user-sa -n feast

# Retrieve and store the token
get_k8s_token("feast-unauthorized-user-sa")

Creating Unauthorized ServiceAccount: feast-unauthorized-user-sa
serviceaccount/feast-unauthorized-user-sa created


'Token Retrieved: ***** (hidden for security)'

In [149]:
!echo "Running Feast RBAC test for Unauthorized User..."
from feast.data_source import PushMode
from feast import FeatureStore
from datetime import datetime
import pandas as pd

try:

    store = FeatureStore(repo_path="client")

    print("\n--- Historical features for training ---")
    fetch_historical_features_entity_df(store, for_batch_scoring=False)

    print("\n--- Historical features for batch scoring ---")
    fetch_historical_features_entity_df(store, for_batch_scoring=True)

    try:
        print("\n--- Load features into online store/materialize_incremental ---")
        feature_views= store.list_feature_views()
        if not feature_views:
            raise PermissionError("No access to feature-views or no feature-views available.")
        store.materialize_incremental(end_date=datetime.now())
    except PermissionError as pe:
        print(f"Permission error: {pe}")
    except Exception as e:
        print(f"An occurred while performing materialize incremental: {e}")

    print("\n--- Online features ---")
    fetch_online_features(store)

    print("\n--- Online features retrieved (instead) through a feature service---")
    fetch_online_features(store, source="feature_service")

    print(
        "\n--- Online features retrieved (using feature service v3, which uses a feature view with a push source---"
    )
    fetch_online_features(store, source="push")

    print("\n--- Simulate a stream event ingestion of the hourly stats df ---")
    event_df = pd.DataFrame.from_dict(
        {
            "driver_id": [1001],
            "event_timestamp": [datetime.now()],
            "created": [datetime.now()],
            "conv_rate": [1.0],
            "acc_rate": [1.0],
            "avg_daily_trips": [1000],
        }
    )
    store.push("driver_stats_push_source", event_df, to=PushMode.ONLINE_AND_OFFLINE)

    print("\n--- Online features again with updated values from a stream push---")
    fetch_online_features(store, source="push")

except Exception as e:
    print(f"An error occurred: {e}")


Running Feast RBAC test for Unauthorized User...

--- Historical features for training ---
Handling connection for 8083
An error occurred while fetching historical features: Permission error:
Permission feast_admin_permission denied execution of ['DESCRIBE'] to FeatureView:driver_hourly_stats: Requires roles ['feast-writer'],Permission feast_user_permission denied execution of ['DESCRIBE'] to FeatureView:driver_hourly_stats: Requires roles ['feast-reader']

--- Historical features for batch scoring ---
An error occurred while fetching historical features: Permission error:
Permission feast_admin_permission denied execution of ['DESCRIBE'] to FeatureView:driver_hourly_stats: Requires roles ['feast-writer'],Permission feast_user_permission denied execution of ['DESCRIBE'] to FeatureView:driver_hourly_stats: Requires roles ['feast-reader']

--- Load features into online store/materialize_incremental ---
Permission error: No access to feature-views or no feature-views available.

--- Onlin

## Setup & Test Admin Feast User (serviceaccount: feast-admin-sa, role: feast-writer)

In [150]:
# Create the ServiceAccount
!echo "Creating ServiceAccount: feast-admin-sa"
!kubectl create serviceaccount feast-admin-sa -n feast

# Assign RoleBinding (Admin Access for Feast)
!echo "Assigning Admin RoleBinding: feast-admin-rolebinding"
!kubectl create rolebinding feast-admin-rolebinding --role=feast-writer --serviceaccount=feast:feast-admin-sa -n feast

# Retrieve and store the token
get_k8s_token("feast-admin-sa")

Creating ServiceAccount: feast-admin-sa
serviceaccount/feast-admin-sa created
Assigning Admin RoleBinding: feast-admin-rolebinding
rolebinding.rbac.authorization.k8s.io/feast-admin-rolebinding created


'Token Retrieved: ***** (hidden for security)'

In [151]:
!echo "Running Feast RBAC test for Admin User..."
from feast.data_source import PushMode
from feast import FeatureStore
from datetime import datetime
import pandas as pd

try:

    store = FeatureStore(repo_path="client")

    print("\n--- Historical features for training ---")
    fetch_historical_features_entity_df(store, for_batch_scoring=False)

    print("\n--- Historical features for batch scoring ---")
    fetch_historical_features_entity_df(store, for_batch_scoring=True)

    try:
        print("\n--- Load features into online store/materialize_incremental ---")
        feature_views= store.list_feature_views()
        if not feature_views:
            raise PermissionError("No access to feature-views or no feature-views available.")
        store.materialize_incremental(end_date=datetime.now())
    except PermissionError as pe:
        print(f"Permission error: {pe}")
    except Exception as e:
        print(f"An occurred while performing materialize incremental: {e}")

    print("\n--- Online features ---")
    fetch_online_features(store)

    print("\n--- Online features retrieved (instead) through a feature service---")
    fetch_online_features(store, source="feature_service")

    print(
        "\n--- Online features retrieved (using feature service v3, which uses a feature view with a push source---"
    )
    fetch_online_features(store, source="push")

    print("\n--- Simulate a stream event ingestion of the hourly stats df ---")
    event_df = pd.DataFrame.from_dict(
        {
            "driver_id": [1001],
            "event_timestamp": [datetime.now()],
            "created": [datetime.now()],
            "conv_rate": [1.0],
            "acc_rate": [1.0],
            "avg_daily_trips": [1000],
        }
    )
    store.push("driver_stats_push_source", event_df, to=PushMode.ONLINE_AND_OFFLINE)

    print("\n--- Online features again with updated values from a stream push---")
    fetch_online_features(store, source="push")

except Exception as e:
    print(f"An error occurred: {e}")


Running Feast RBAC test for Admin User...

--- Historical features for training ---
Handling connection for 8083
Handling connection for 8081
   driver_id           event_timestamp  label_driver_reported_satisfaction  \
0       1001 2021-04-12 10:59:42+00:00                                   1   
1       1002 2021-04-12 08:12:10+00:00                                   5   
2       1003 2021-04-12 16:40:26+00:00                                   3   

   val_to_add  val_to_add_2  conv_rate  acc_rate  avg_daily_trips  \
0           1            10   0.274426  0.314969              971   
1           2            20   0.742959  0.215825              723   
2           3            30   0.466147  0.412917              247   

   conv_rate_plus_val1  conv_rate_plus_val2  
0             1.274426            10.274426  
1             2.742959            20.742959  
2             3.466147            30.466147  

--- Historical features for batch scoring ---
Handling connection for 8081
   drive

0it [00:00, ?it/s]


driver_hourly_stats_fresh from 2025-02-27 16:14:47-05:00 to 2025-02-27 11:45:57-05:00:
Handling connection for 8081


0it [00:00, ?it/s]



--- Online features ---
Handling connection for 8082
acc_rate  :  [0.4686085283756256, 0.2513408660888672]
conv_rate_plus_val1  :  [1000.7328559160233, 1001.82472884655]
conv_rate_plus_val2  :  [2000.7328559160233, 2002.82472884655]
driver_id  :  [1001, 1002]

--- Online features retrieved (instead) through a feature service---
Handling connection for 8082
conv_rate  :  [0.7328559160232544, 0.8247288465499878]
conv_rate_plus_val1  :  [1000.7328559160233, 1001.82472884655]
conv_rate_plus_val2  :  [2000.7328559160233, 2002.82472884655]
driver_id  :  [1001, 1002]

--- Online features retrieved (using feature service v3, which uses a feature view with a push source---
Handling connection for 8082
acc_rate  :  [0.4686085283756256, 0.2513408660888672]
avg_daily_trips  :  [830, 20]
conv_rate  :  [0.7328559160232544, 0.8247288465499878]
conv_rate_plus_val1  :  [1000.7328559160233, 1001.82472884655]
conv_rate_plus_val2  :  [2000.7328559160233, 2002.82472884655]
driver_id  :  [1001, 1002]

--- 

 **Note:**
Currently, remote materialization not available in Feast when using the Remote Client so the above error showing in that step.

 Workaround: Consider using running it from pod like
  
 `kubectl exec deploy/feast-sample-kubernetes-auth -itc online -- bash -c 'feast materialize-incremental $(date -u +"%Y-%m-%dT%H:%M:%S")`


## Uninstall

### Uninstall the Operator and all Feast related objects##

In [110]:
!kubectl delete -f ../../infra/feast-operator/config/samples/v1alpha1_featurestore_kubernetes_auth.yaml
!kubectl delete -f ../../infra/feast-operator/dist/install.yaml

featurestore.feast.dev "sample-kubernetes-auth" deleted
namespace "feast-operator-system" deleted
customresourcedefinition.apiextensions.k8s.io "featurestores.feast.dev" deleted
serviceaccount "feast-operator-controller-manager" deleted
role.rbac.authorization.k8s.io "feast-operator-leader-election-role" deleted
clusterrole.rbac.authorization.k8s.io "feast-operator-featurestore-editor-role" deleted
clusterrole.rbac.authorization.k8s.io "feast-operator-featurestore-viewer-role" deleted
clusterrole.rbac.authorization.k8s.io "feast-operator-manager-role" deleted
clusterrole.rbac.authorization.k8s.io "feast-operator-metrics-auth-role" deleted
clusterrole.rbac.authorization.k8s.io "feast-operator-metrics-reader" deleted
rolebinding.rbac.authorization.k8s.io "feast-operator-leader-election-rolebinding" deleted
clusterrolebinding.rbac.authorization.k8s.io "feast-operator-manager-rolebinding" deleted
clusterrolebinding.rbac.authorization.k8s.io "feast-operator-metrics-auth-rolebinding" deleted

## Uninstall Client Related Objects

In [146]:
!echo "Deleting RoleBindings..."
!kubectl delete rolebinding feast-user-rolebinding -n feast --ignore-not-found
!kubectl delete rolebinding feast-admin-rolebinding -n feast --ignore-not-found

!echo "Deleting ServiceAccounts..."
!kubectl delete serviceaccount feast-user-sa -n feast --ignore-not-found
!kubectl delete serviceaccount feast-admin-sa -n feast --ignore-not-found
!kubectl delete serviceaccount feast-unauthorized-user-sa -n feast --ignore-not-found


Deleting RoleBindings...
rolebinding.rbac.authorization.k8s.io "feast-user-rolebinding" deleted
rolebinding.rbac.authorization.k8s.io "feast-admin-rolebinding" deleted
Deleting ServiceAccounts...
serviceaccount "feast-user-sa" deleted
serviceaccount "feast-admin-sa" deleted
serviceaccount "feast-unauthorized-user-sa" deleted


Ensure everything has been removed, or is in the process of being terminated.

In [118]:
!kubectl get all -n feast


No resources found in feast namespace.


In [113]:
for name, process in port_forward_processes.items():
    process.terminate()
    print(f"Stopped port forwarding for {name}")

Stopped port forwarding for offline_store
Stopped port forwarding for online_store
Stopped port forwarding for registry
